In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rc('font', size=14)
np.set_printoptions(precision=5, suppress=True)

# ---------------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------------

airfoil_file = "s7055.dat"

# Wing / section data
chord = 0.265
skin_thickness = 0.5e-3

# Stringer geometry
stringer_thickness = 0.5e-3
stringer_length = 20e-3

# Number of stringers to add.
# Change this to match your section.
num_stringers = 7

# Stringer positions as normalized chordwise coordinates (x/c)
#
# Enter upper and lower stringer locations separately.
#
# Example:
# upper_stringer_x_norm = np.array([0.10, 0.30, 0.70, 0.90])
# lower_stringer_x_norm = np.array([0.10, 0.30, 0.70, 0.90])
#
# x/c = 0 -> Leading Edge
# x/c = 1 -> Trailing Edge

upper_stringer_x_norm = np.array([0.1, 0.3, 0.7, 0.9])
lower_stringer_x_norm = np.array([0.2,0.4,0.7])

# Vertical shear force used only to obtain the shear-center moment.
# The resulting shear-center location is independent of its magnitude.
V = 1.0  # N

spar_location_norm = 0.3174
spar_location = spar_location_norm*chord
theo_spar_height = 0.028
flange_length = 0.03
flange_thickness = 1e-3
web_thickness = 1e-3

In [11]:
# LOAD AIRFOIL
# ---------------------------------------------------------------------

airfoil = np.loadtxt(airfoil_file) * chord

# Selig coordinates contain a duplicate trailing-edge point.
# Remove the final duplicate if it is coincident with the first point.
if np.allclose(airfoil[0], airfoil[-1]):
    airfoil = airfoil[:-1]

x = airfoil[:, 0]
y = airfoil[:, 1]

n = len(x)

# Closed panels
x_next = np.roll(x, -1)
y_next = np.roll(y, -1)

dx = x_next - x
dy = y_next - y
ds = np.hypot(dx, dy)

# Panel midpoints
x_mid = 0.5 * (x + x_next)
y_mid = 0.5 * (y + y_next)


In [12]:
# ---------------------------------------------------------------------
# 1. SKIN AREA IDEALISATION
# ---------------------------------------------------------------------

A_skin_panel = skin_thickness * ds
A_skin = np.sum(A_skin_panel)

x_centroid_skin = np.sum(A_skin_panel * x_mid) / A_skin
y_centroid_skin = np.sum(A_skin_panel * y_mid) / A_skin

print(f"Skin area       = {A_skin:.6e} m^2")
print(f"Skin centroid x = {x_centroid_skin:.6e} m")
print(f"Skin centroid y = {y_centroid_skin:.6e} m")

yc_skin = y - y_centroid_skin

# Check for points too close to the reference axis.
if np.any(np.abs(yc_skin) < 1e-10):
    raise ValueError(
        "A boom lies essentially on the skin centroidal axis. "
        "The standard boom-area formula becomes singular. "
        "Use a different boom/reference discretisation."
    )

l_prev = np.roll(ds, 1)
l_next = ds

y_prev = np.roll(yc_skin, 1)
y_next = np.roll(yc_skin, -1)

B_skin = (
    skin_thickness / 6.0
    * (
        l_prev * (2.0 + y_prev / yc_skin)
        + l_next * (2.0 + y_next / yc_skin)
    )
)


Skin area       = 2.695333e-04 m^2
Skin centroid x = 1.311096e-01 m
Skin centroid y = 6.145572e-03 m


In [13]:

# ---------------------------------------------------------------------
# 3. ADD STRINGER BOOM AREAS
# ---------------------------------------------------------------------

B_stringer = np.zeros(n)

# Stringer boom area
A_stringer = stringer_thickness * stringer_length


def find_surface_index(x_norm, surface):
    """
    Find the S7055 coordinate closest to the requested x/c
    on either the upper or lower surface.
    """

    x_target = x_norm * chord

    if surface == "upper":
        candidates = np.where(y >= 0)[0]
    elif surface == "lower":
        candidates = np.where(y <= 0)[0]
    else:
        raise ValueError("surface must be 'upper' or 'lower'")

    if len(candidates) == 0:
        raise ValueError(f"No points found on {surface} surface.")

    idx = candidates[np.argmin(np.abs(x[candidates] - x_target))]

    return idx


# -------------------------
# Upper stringers
# -------------------------

upper_stringer_indices = []

for x_norm in upper_stringer_x_norm:

    if not 0 <= x_norm <= 1:
        raise ValueError(
            f"Upper stringer x/c = {x_norm} is outside 0 <= x/c <= 1."
        )

    idx = find_surface_index(x_norm, "upper")

    B_stringer[idx] += A_stringer

    upper_stringer_indices.append(idx)


# -------------------------
# Lower stringers
# -------------------------

lower_stringer_indices = []

for x_norm in lower_stringer_x_norm:

    if not 0 <= x_norm <= 1:
        raise ValueError(
            f"Lower stringer x/c = {x_norm} is outside 0 <= x/c <= 1."
        )

    idx = find_surface_index(x_norm, "lower")

    B_stringer[idx] += A_stringer

    lower_stringer_indices.append(idx)


# Total boom area
B_total = B_skin + B_stringer


In [14]:
upper_spar_index = find_surface_index(spar_location_norm, "upper")
lower_spar_index = find_surface_index(spar_location_norm, "lower")

upper_flange_location = spar_location + flange_length
lower_flange_location = spar_location + flange_length

upper_flange_index = find_surface_index(upper_flange_location / chord, "upper")
lower_flange_index = find_surface_index(lower_flange_location / chord, "lower")

print(f"Upper spar index: {upper_spar_index}")
print(f"Upper flange index: {upper_flange_index}")
print(f"Lower spar index: {lower_spar_index}")
print(f"Lower flange index: {lower_flange_index}")

Upper spar index: 25
Upper flange index: 22
Lower spar index: 57
Lower flange index: 60


In [15]:
spar_height = y[upper_spar_index] - y[lower_spar_index]
print(f"Spar height: {spar_height}" , 
      f"theoretical spar height: {theo_spar_height}" )

Spar height: 0.0277455 theoretical spar height: 0.028


In [ ]:
dl 

In [17]:
Boom_upper_spar = web_thickness*spar_height*(2+y[lower_spar_index]/y[upper_spar_index])/6 
    + flange_thickness
Boom_lower_spar = web_thickness*spar_height*(2+y[upper_spar_index]/y[lower_spar_index])/6



IndentationError: unexpected indent (4076203272.py, line 2)